In [18]:
import os
import warnings
warnings.filterwarnings("ignore")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_groq import ChatGroq


In [21]:
from dotenv import load_dotenv
load_dotenv()

http_proxy = os.getenv('http_proxy')
https_proxy = os.getenv('https_proxy')
HTTP_PROXY = os.getenv('HTTP_PROXY')
HTTPS_PROXY = os.getenv('HTTPS_PROXY')


os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")



In [22]:
def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents


pdf_path = "/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/AI_Engineering.pdf"

docs = load_pdf(pdf_path)

print("Total pages:", len(docs))
print("Sample metadata:", docs[0].metadata)


Total pages: 535
Sample metadata: {'producer': 'Antenna House PDF Output Library 2.6.0 (Linux64)', 'creator': 'AH CSS Formatter V6.0 MR2 for Linux64 : 6.0.2.5372 (2012/05/16 18:26JST)', 'creationdate': '2024-12-04T13:39:11+00:00', 'author': 'Chip Huyen;', 'moddate': '2024-12-04T09:21:26-05:00', 'title': 'AI Engineering', 'trapped': '/False', 'ebx_publisher': "O'Reilly Media", 'source': '/nmhs2/hari/work/UAV/Projects/rag_pipeline/pdfs/AI_Engineering.pdf', 'total_pages': 535, 'page': 0, 'page_label': 'Cover'}


In [23]:
def chunk_documents(docs, chunk_size=1000, chunk_overlap=150):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_documents(docs)
    return chunks


documents = chunk_documents(docs)

print("Total chunks:", len(documents))


Total chunks: 1452


In [6]:
documents=chunk_data(docs=docs)
len(documents)

535

In [24]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# test
vec = embeddings.embed_query("What is AI?")
print("Vector size:", len(vec))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 585.86it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vector size: 384


In [25]:
vector_store = FAISS.from_documents(documents, embeddings)

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)


In [26]:
llm = ChatGroq(
    model_name="openai/gpt-oss-120b", 
    temperature=0
)


In [27]:
custom_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are an AI research assistant.

Answer ONLY using the provided context.
If the answer is not found, say:
"I could not find the answer in the document."

Context:
{context}

Question:
{question}

Answer:
"""
)


In [28]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True,
    chain_type="stuff",
    chain_type_kwargs={"prompt": custom_prompt}
)



In [29]:
query = "What is fine tuning?"

result = qa_chain.invoke({"query": query})

print("Answer:\n", result["result"])


Answer:
 Fine‑tuning is the process of updating a model’s parameters on a downstream task after it has been pre‑trained. During fine‑tuning, some or all of the model’s parameters are made trainable (i.e., they can be updated), while the remaining parameters are kept frozen. This adapts the pre‑trained model to a new task without changing the inference‑only behavior of the model.


In [31]:
while True:
    question = input("\nAsk a question (type 'exit'): ")

    if question.lower() == "exit":
        break

    result = qa_chain.invoke({"query": question})

    print("\nAnswer:\n", result["result"])



Answer:
 Fine‑tuning is the process of taking a pre‑trained model and updating some or all of its parameters on a new, downstream task. During fine‑tuning, trainable parameters are adjusted via back‑propagation (while other parameters may be frozen), allowing the model to adapt its knowledge to the specific data or application.
